# On-Disk Inductive Learning: Efficient Training on Large Datasets

This tutorial demonstrates TopoBench's **on-disk preprocessing** for training on large inductive datasets that exceed available RAM.

---

## 📚 **NEW: Two-Part Tutorial Series**

**This tutorial has been restructured into two focused parts for better learning:**

### 🚀 **Part 1: Getting Started** (15-20 min)
**File:** `tutorial_ondisk_inductive_part1_getting_started.ipynb`

**Learn:**
- ✅ When and why to use on-disk preprocessing
- ✅ Basic workflow from dataset to training
- ✅ Constant memory usage regardless of dataset size
- ✅ Complete end-to-end example

**Start here if:** You're new to on-disk preprocessing

---

### ⚡ **Part 2: Advanced Techniques** (20-25 min)
**File:** `tutorial_ondisk_inductive_part2_advanced.ipynb`

**Learn:**
- ✅ DAG-based incremental caching (2-3× faster iteration)
- ✅ Storage backend options (files vs mmap)
- ✅ Parallel processing (3-4× speedup)
- ✅ Best practices for research workflows

**Start here if:** You completed Part 1 and want to optimize your workflow

---

## 📋 Original Tutorial Below

**Note:** The content below provides a comprehensive single-page reference. For structured learning, we recommend the two-part series above.

---

## 🎯 What You'll Learn

1. ✅ Create custom datasets for on-disk processing
2. ✅ Apply topological transforms (liftings) with constant memory usage
3. ✅ Leverage transform caching for fast experimentation
4. ✅ Train models on **hypergraph** and **simplicial** structures
5. ✅ Scale to datasets that would cause OOM with in-memory approaches

## 📋 Table of Contents

- [1. Why On-Disk?](#section1)
- [2. Dataset Creation](#section2)
- [3. Loader Implementation](#section3)
- [4. On-Disk Preprocessing](#section4)
- [5. Data Loading & Splits](#section5)
- [6. Model Training - Hypergraph](#section6)
- [7. Model Training - Simplicial (Alternative)](#section7)
- [8. Performance & Best Practices](#section8)
- [9. Summary](#section9)

---

## 1. Why On-Disk? <a id="section1"></a>

### The Problem: Memory Explosion 💥

Traditional in-memory preprocessing loads **all topological structures into RAM at once**:

```
Example: 5,000 graphs × 100 nodes × 200 edges/triangles ≈ 6GB RAM
```

**Result**: Out-of-memory (OOM) errors before training even starts!

### The Solution: Streaming to Disk 💾

On-disk preprocessing processes graphs **one-by-one** and streams results to disk:

- **Constant memory**: ~50-100MB regardless of dataset size
- **All transforms supported**: Liftings, features, preprocessing - everything works
- **Persistent caching**: Reuse processed data across experiments
- **Scalable**: Limited only by disk space, not RAM

### When to Use On-Disk ✓

Use on-disk preprocessing when:
- ✅ Dataset has **> 1,000 graphs**
- ✅ Graphs have **> 50 nodes** or high degree  
- ✅ Using **topological liftings** (simplicial, hypergraph, cell)
- ✅ Available **RAM < 8GB** or working on shared systems

### Performance Trade-offs ⚖️

| Aspect | In-Memory | On-Disk |
|--------|-----------|---------|
| **Memory** | O(N × D²) | **O(1) constant** |
| **Preprocessing** | All at once | Stream to disk |
| **Training speed** | Baseline | ~1.2× slower (disk I/O) |
| **Max dataset size** | Limited by RAM | **Limited by disk** |
| **Transform caching** | None | **✅ Persistent** |

💡 **Key insight**: Small training slowdown (disk I/O) vs. enabling training on datasets that would otherwise be impossible!

---

### Prerequisites

Install required packages:

```bash
pip install torch torch-geometric networkx omegaconf pytorch-lightning toponetx topomodelx
```

💡 **Tip**: See `requirements.txt` in the TopoBench repo for exact versions.

---

## 2. Dataset Creation <a id="section2"></a>

Create your dataset by inheriting from `InMemoryDataset`. This is the **source dataset** before applying topological transforms.

Follow the standard TopoBench pattern:

In [2]:
import networkx as nx
import torch
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.io import fs
from omegaconf import DictConfig

class MyLargeInductiveDataset(InMemoryDataset):
    """Custom large inductive dataset.
    
    This creates a source dataset. On-disk preprocessing
    will handle the topological structures efficiently.
    """
    
    def __init__(self, root, name, parameters: DictConfig):
        self.name = name
        self.parameters = parameters
        super().__init__(root)
        
        # Load processed data
        out = fs.torch_load(self.processed_paths[0])
        if len(out) == 4:
            data, self.slices, self.sizes, data_cls = out
            self.data = data_cls.from_dict(data) if isinstance(data, dict) else data
        else:
            data, self.slices, self.sizes = out
            self.data = data
    
    @property
    def raw_file_names(self):
        return []
    
    @property
    def processed_file_names(self):
        return "data.pt"
    
    def download(self):
        pass  # Implement if downloading from external source
    
    def process(self):
        """Generate your graphs here."""
        data_list = []
        
        # Example: Generate synthetic graphs (replace with your data)
        for i in range(self.parameters.num_graphs):
            G = nx.watts_strogatz_graph(
                n=self.parameters.nodes_per_graph,
                k=self.parameters.degree,
                p=0.3,
                seed=42+i
            )
            
            # Convert to PyG Data
            edges = list(G.edges())
            edge_index = torch.tensor(edges, dtype=torch.long).t()
            edge_index = torch.cat([edge_index, edge_index[[1, 0]]], dim=1)  # Undirected
            
            x = torch.randn(G.number_of_nodes(), self.parameters.num_features)
            y = torch.randint(0, self.parameters.num_classes, (1,))
            
            data = Data(x=x, edge_index=edge_index, y=y, num_nodes=G.number_of_nodes())
            data_list.append(data)
        
        # Collate and save
        self.data, self.slices = self.collate(data_list)
        fs.torch_save(
            (self._data.to_dict(), self.slices, {}, self._data.__class__),
            self.processed_paths[0]
        )

---

## 3. Loader Implementation <a id="section3"></a>

Create a loader following TopoBench's `AbstractLoader` pattern:

In [3]:
from topobench.data.loaders.base import AbstractLoader

class MyLargeInductiveLoader(AbstractLoader):
    """Loader for custom inductive dataset."""
    
    def __init__(self, parameters: DictConfig):
        super().__init__(parameters)
    
    def load_dataset(self):
        dataset = MyLargeInductiveDataset(
            root=str(self.root_data_dir),
            name=self.parameters.data_name,
            parameters=self.parameters
        )
        return dataset

/home/tgrapentin/personal/tdl/Topo2/TopoBench/venv/lib/python3.12/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


---

## 4. On-Disk Preprocessing <a id="section4"></a>

Now we load the dataset and apply **on-disk preprocessing** with topological transforms.

### 4.1 Load Source Dataset

In [18]:
from omegaconf import OmegaConf
from topobench.data.preprocessor import OnDiskInductivePreprocessor

# Configure your dataset
loader_config = OmegaConf.create({
    "data_dir": "./data/MyLargeDataset",
    "data_name": "MyLargeDataset", # Change this name or delete data when re-running with different params
    "num_graphs": 50, # 5000,
    "nodes_per_graph": 20, # 80,
    "degree": 4, # 15,
    "num_features": 16,
    "num_classes": 5
})

print(loader_config)

# Load source dataset
loader = MyLargeInductiveLoader(loader_config)
dataset, dataset_dir = loader.load()
print(f"Loaded {len(dataset)} graphs")

# Configure topological transforms (liftings)
# transforms_config = OmegaConf.create({
#     "clique_lifting": {
#         "transform_type": "lifting",
#         "transform_name": "SimplicialCliqueLifting",
#         "complex_dim": 2  # Include up to triangles
#     }
# })

transforms_config = OmegaConf.create({
    "khop_lifting": {
        "transform_type": "lifting",
        "transform_name": "HypergraphKHopLifting",
        "k_value": 2,
        "signed": False
    }
})

# Create on-disk preprocessor with transforms
ondisk_dataset_preprocessor = OnDiskInductivePreprocessor(
    dataset=dataset,
    data_dir=dataset_dir,
    transforms_config=transforms_config,  # Transforms applied during preprocessing!
    force_reload=False  # Reuses cached data if config unchanged
)

print(f"✓ On-disk preprocessing complete")
print(f"  - Samples: {len(ondisk_dataset_preprocessor)}")
print(f"  - Memory: Constant (~50-100MB)")
print(f"  - Transforms: Cached on disk for reuse")

{'data_dir': './data/MyLargeDataset', 'data_name': 'MyLargeDataset', 'num_graphs': 50, 'nodes_per_graph': 20, 'degree': 4, 'num_features': 16, 'num_classes': 5}
Loaded 50 graphs
✓ On-disk preprocessing complete
  - Samples: 50
  - Memory: Constant (~50-100MB)
  - Transforms: Cached on disk for reuse


### 4.2 Alternative: Factory Function (Recommended!)

For simpler code, use the `create_preprocessor()` factory which automatically selects the best preprocessor:

```python
from topobench.data.preprocessor import create_preprocessor

preprocessor = create_preprocessor(
    dataset=dataset,
    data_dir="./data/MyLargeDataset/processed",
    transforms_config=transforms_config,
    mode="auto",  # Automatically selects in-memory or on-disk
    available_ram_gb=4  # Optional: specify available RAM
)
```

**Benefits**:
- Unified interface for all preprocessor types
- Automatic selection based on available RAM
- Cleaner, more maintainable code

For this tutorial, we'll use the explicit `OnDiskInductivePreprocessor` to show what's happening under the hood.

In [5]:
from topobench.data.preprocessor import create_preprocessor

# Unified interface - same for in-memory or on-disk!
preprocessor = create_preprocessor(
    dataset=dataset,
    data_dir="./data/MyLargeDataset/processed/auto",
    transforms_config=transforms_config,
    mode="ondisk",  # Options: "auto", "inmemory", "ondisk". Use "auto" for automatic selection based on available RAM
    available_ram_gb=4,  # Specify available RAM in GB. If left empty, it will be estimated
)

print(f"✓ Preprocessor created: {type(preprocessor).__name__}")
# Will be OnDiskInductivePreprocessor for large datasets

✓ Preprocessor created: OnDiskInductivePreprocessor


---

## 5. Data Loading & Splits <a id="section5"></a>

### 5.1 Create Dataset Splits

Split your preprocessed data into train/val/test sets:

In [19]:
from topobench.data.utils import load_inductive_splits

# Configure splits
split_config = OmegaConf.create({
    "learning_setting": "inductive",
    "split_type": "random",
    "data_seed": 0,
    "data_split_dir": "./data/MyLargeDataset/splits/",
    "train_prop": 0.5,
    "val_prop": 0.25,
})

# Load splits (built-in support)
train, val, test = ondisk_dataset_preprocessor.load_dataset_splits(split_config)

print(f"Splits created:")
print(f"  - Train: {len(train)} samples")
print(f"  - Val: {len(val)} samples")
print(f"  - Test: {len(test)} samples")

Splits created:
  - Train: 25 samples
  - Val: 12 samples
  - Test: 13 samples


### 5.2 Create DataLoader

Use standard TopoBench `TBDataloader`:

In [20]:
from topobench.dataloader import TBDataloader

# Create dataloader (works identically to in-memory)
datamodule = TBDataloader(
    dataset_train=train,
    dataset_val=val,
    dataset_test=test,
    batch_size=32,
    num_workers=0  # Set >0 for multi-process loading
)

print("✓ Dataloader ready")

✓ Dataloader ready


---

## 6. Model Training - Hypergraph <a id="section6"></a>

This section demonstrates training with **hypergraph structures** created by `HypergraphKHopLifting`.

### Key Concepts:
- **Hypergraph**: Nodes + hyperedges (hyperedges can connect > 2 nodes)
- **Model**: EDGNN (Equivariant Dynamic Graph Neural Network)
- **Data attributes**: `incidence_hyperedges`, `x_0` (node features)
- **Dimensions**: 0 (nodes) and 1 (hyperedges)

In [22]:

from lightning import Trainer
from topobench.model import TBModel
from topobench.nn.readouts import PropagateSignalDown
from topobench.loss import TBLoss
from topobench.optimizer import TBOptimizer
from topobench.evaluator.evaluator import TBEvaluator
from topobench.nn.encoders import AllCellFeatureEncoder

# Model configuration
HIDDEN_DIM = 64
OUT_CHANNELS = 5
NUM_FEATURES = 16

# =============================================================================
# HYPERGRAPH APPROACH (for HypergraphKHopLifting)
# =============================================================================
from topobench.nn.backbones.hypergraph import EDGNN
from topobench.nn.wrappers.hypergraph import HypergraphWrapper

# Create feature encoder
feature_encoder = AllCellFeatureEncoder(
    in_channels=[NUM_FEATURES],  # Only node features for hypergraph
    out_channels=HIDDEN_DIM
)

# Create EDGNN backbone for hypergraphs
backbone = EDGNN(
    num_features=HIDDEN_DIM,
    input_dropout=0.2,
    dropout=0.2,
    All_num_layers=2
)

# Readout configuration
readout_config = {
    "readout_name": "PropagateSignalDown",
    "num_cell_dimensions": 1,  # Hypergraph: nodes (0) and hyperedges (1)
    "hidden_dim": HIDDEN_DIM,
    "out_channels": OUT_CHANNELS,
    "task_level": "node",
    "pooling_type": "sum",
}

# Wrapper factory for hypergraph
def wrapper(**factory_kwargs):
    def factory(backbone):
        return HypergraphWrapper(backbone, **factory_kwargs)
    return factory

wrapper_config = {
    "out_channels": HIDDEN_DIM,
    "num_cell_dimensions": 1,  # Hypergraph has 2 dimensions: 0 and 1
}

# =============================================================================
# SIMPLICIAL APPROACH (for SimplicialCliqueLifting)
# =============================================================================
# from topomodelx.nn.simplicial.scn2 import SCN2
# from topobench.nn.wrappers.simplicial import SCNWrapper

# # Create feature encoder for simplicial
# feature_encoder = AllCellFeatureEncoder(
#     in_channels=[NUM_FEATURES, NUM_FEATURES, NUM_FEATURES],  # Node, edge, triangle features
#     out_channels=HIDDEN_DIM
# )

# # Create SCN2 backbone for simplicial complexes
# backbone = SCN2(
#     in_channels_0=HIDDEN_DIM,
#     in_channels_1=HIDDEN_DIM,
#     in_channels_2=HIDDEN_DIM
# )

# # Readout configuration
# readout_config = {
#     "readout_name": "PropagateSignalDown",
#     "num_cell_dimensions": 2,  # Simplicial: nodes (0), edges (1), triangles (2)
#     "hidden_dim": HIDDEN_DIM,
#     "out_channels": OUT_CHANNELS,
#     "task_level": "node",
#     "pooling_type": "sum",
# }

# # Wrapper factory for simplicial
# def wrapper(**factory_kwargs):
#     def factory(backbone):
#         return SCNWrapper(backbone, **factory_kwargs)
#     return factory

# wrapper_config = {
#     "out_channels": HIDDEN_DIM,
#     "num_cell_dimensions": 2,  # Simplicial has 3 dimensions: 0, 1, 2
# }

# =============================================================================
# Common configuration (same for both approaches)
# =============================================================================

readout = PropagateSignalDown(**readout_config)

# Evaluator configuration
evaluator_config = {
    "task": "classification",
    "num_classes": OUT_CHANNELS,
    "metrics": ["accuracy", "precision", "recall"]
}

evaluator = TBEvaluator(**evaluator_config)

# Loss configuration
loss = TBLoss(dataset_loss={
    "task": "classification",
    "loss_type": "cross_entropy"
})

# Optimizer configuration
optimizer = TBOptimizer(
    optimizer_id="Adam",
    parameters={"lr": 0.01}
)

# Create wrapper
backbone_wrapper = wrapper(**wrapper_config)

# Create TopoBench model
model = TBModel(
    backbone=backbone,
    backbone_wrapper=backbone_wrapper,
    readout=readout,
    loss=loss,
    feature_encoder=feature_encoder,
    evaluator=evaluator,
    optimizer=optimizer,
    compile=False,
)

# Train with Lightning
trainer = Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    enable_progress_bar=True
)

trainer.fit(model, datamodule)

print("✅ Training complete!")
print("   Memory stayed constant throughout training.")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name            | Type                  | Params | Mode 
------------------------------------------------------------------
0 | feature_encoder | AllCellFeatureEncoder | 1.1 K  | train
1 | backbone        | HypergraphWrapper     | 29.2 K | train
2 | readout         | PropagateSignalDown   | 325    | train
3 | val_acc_best    | MeanMetric            | 0      | train
------------------------------------------------------------------
30.6 K    Trainable params
0         Non-trainable params
30.6 K    Total params
0.123     Total estimated model params size (MB)
37        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


✅ Training complete!
   Memory stayed constant throughout training.


---

## 7. Model Training - Simplicial (Alternative) <a id="section7"></a>

**Alternative approach**: Train with **simplicial complexes** using `SimplicialCliqueLifting`.

### To switch to simplicial:
1. In Cell 8: Comment out `HypergraphKHopLifting`, uncomment `SimplicialCliqueLifting`
2. Rerun preprocessing with `force_reload=True`
3. In Cell 16: Comment out hypergraph section, uncomment simplicial section

###Key Differences:
| Aspect | Hypergraph | Simplicial |
|--------|------------|------------|
| **Lifting** | `HypergraphKHopLifting` | `SimplicialCliqueLifting` |
| **Model** | EDGNN | SCN2 |
| **Wrapper** | `HypergraphWrapper` | `SCNWrapper` |
| **Dimensions** | 2 (nodes, hyperedges) | 3 (nodes, edges, triangles) |
| **num_cell_dimensions** | 1 | 2 |

See `TUTORIAL_HYPERGRAPH_VS_SIMPLICIAL.md` for detailed comparison!

---

## 8. Performance & Best Practices <a id="section8"></a>

### Memory & Scalability 📊

| Dataset Size | In-Memory RAM | On-Disk RAM | Speed Impact |
|--------------|---------------|-------------|--------------|
| 100 graphs | ~300MB | ~80MB | Negligible |
| 1,000 graphs | ~2GB | ~80MB | +10% slower |
| 5,000 graphs | ~10GB (OOM!) | ~80MB | +20% slower |
| 10,000+ graphs | Not possible | ~80MB | +25% slower |

### Best Practices ✓

1. **Test small first**: Start with 50-100 graphs to verify your pipeline
2. **Monitor disk space**: Processed data ≈ 2-5× original size
3. **Use SSD**: Significantly reduces I/O overhead during training
4. **Cache reuse**: Same transform config = instant load from cache
5. **Force reload**: Set `force_reload=True` if you change transform parameters
6. **Batch size**: Larger batches reduce I/O frequency (try 32-64)

### Transform Caching Example

```python
# First run: Processes all graphs (takes time)
preprocessor_v1 = OnDiskInductivePreprocessor(
    dataset=dataset,
    data_dir="./processed",
    transforms_config=config,  
    force_reload=False
)

# Second run with SAME config: Instant load! ⚡
preprocessor_v2 = OnDiskInductivePreprocessor(
    dataset=dataset,
    data_dir="./processed",
    transforms_config=config,  # Same → cached
    force_reload=False
)
```

### Hardware Recommendations

| Component | Minimum | Recommended |
|-----------|---------|-------------|
| **RAM** | 4GB | 8GB+ |
| **Disk** | HDD (works) | **SSD** (fast) |
| **Disk Space** | 2× dataset size | 5× dataset size |
| **CPU** | 2 cores | 4+ cores |

---

## 9. Summary <a id="section9"></a>

### What You Learned 🎓

1. ✅ **Dataset Creation**: Custom `InMemoryDataset` → `AbstractLoader` pattern
2. ✅ **On-Disk Preprocessing**: Process graphs one-by-one with constant memory
3. ✅ **Topological Transforms**: Apply liftings (hypergraph, simplicial) efficiently
4. ✅ **Transform Caching**: Reuse processed data across experiments
5. ✅ **Model Training**: Train on both hypergraph (EDGNN) and simplicial (SCN2) structures
6. ✅ **Scalability**: Handle datasets that would cause OOM with in-memory approaches

### Key Takeaways 🔑

- **Memory**: On-disk uses O(1) constant memory vs. O(N × D²) for in-memory
- **Speed**: ~1.2× slower training is worth it for large datasets
- **Caching**: Transform results persist across runs - saves hours of preprocessing
- **Flexibility**: Works with all TopoBench transforms and models

### When to Use On-Disk ✓

Use on-disk preprocessing when:
- Dataset has **> 1,000 graphs**
- Graphs have **> 50 nodes** or complex structure
- Using **topological liftings** (memory intensive)
- Available **RAM < 8GB** or working on shared systems
- Want **persistent caching** of expensive transforms

---

## 🚀 Recommended Next Steps

### **For Structured Learning:**

👉 **New to on-disk preprocessing?**  
Start with: `tutorial_ondisk_inductive_part1_getting_started.ipynb`
- Focused 15-minute introduction
- Step-by-step guidance
- Copy-paste ready code

👉 **Want to optimize your workflow?**  
Continue with: `tutorial_ondisk_inductive_part2_advanced.ipynb`
- DAG-based caching (2-3× faster iteration)
- Storage backend options (files vs mmap)
- Parallel processing techniques
- Research workflow best practices

### **Other Resources:**

- **`tutorial_ondisk_transductive.ipynb`**: Large single-graph learning (like OGBN-products)
- **`README_DAG_CACHING.md`**: Technical deep-dive on incremental caching
- **`SPEED_VS_COMPRESSION_TRADEOFF.md`**: Files vs mmap backend comparison
- **`tutorial_lifting.ipynb`**: Deep dive into topological transforms  
- **`tutorial_model.ipynb`**: Create custom models and architectures
- **`OGBN_PRODUCTS_GUIDE.md`**: Real-world example with 2.4M nodes

### Need Help? 📚

- **Documentation**: https://github.com/pyt-team/TopoBench
- **Issues**: https://github.com/pyt-team/TopoBench/issues

---

**Happy training! 🎉**

*Remember: With on-disk preprocessing, the only limit is your disk space, not your RAM!*